# RMIT Hackathon 2026 - Challenge 1: Multilingual Prompt-Attack Classification
**Team:** RMIT_SGS_ComGa_C1

**Chosen model:** `FacebookAI/xlm-roberta-large` (encoder) - https://www.kaggle.com/models/aisuko/xlm-roberta-large/Transformers/default

Approach: fine-tune with PyTorch, Gradient Checkpointing + Gradient Accumulation to guarantee NO CUDA OOM on T4 (15GB), class-weighted cross-entropy for imbalanced classes (Macro-F1), language tag prefix, fp16, and validation logit-bias tuning.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, random, time, gc
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# Dọn sạch bộ nhớ GPU trước khi chạy
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

NGPU = torch.cuda.device_count()
DEVICE = torch.device('cuda' if NGPU else 'cpu')
print('CUDA Available:', torch.cuda.is_available(), '| GPUs:', NGPU)
if NGPU:
    used = torch.cuda.memory_allocated() / 1e9
    print(f'GPU memory already in use at start: {used:.2f} GB')
    assert used < 1.0, 'GPU memory is occupied by an earlier run -> Run > Restart Session, then Run All'
if NGPU:
    print('Device:', torch.cuda.get_device_name(0))

# ---- paths ----
MODEL_PATH = None
for p in ['/kaggle/input/models/aisuko/xlm-roberta-large/transformers/default/1',
          '/kaggle/input/xlm-roberta-large/transformers/default/1']:
    if os.path.exists(p):
        MODEL_PATH = p; break
if MODEL_PATH is None:
    found = glob.glob('/kaggle/input/**/xlm-roberta-large/**/config.json', recursive=True)
    MODEL_PATH = os.path.dirname(found[0]) if found else 'xlm-roberta-large'
DATA_DIR = next(d for d in ['/kaggle/input/competitions/rmit-hackathon-2026', '/kaggle/input/rmit-hackathon-2026']
                if os.path.exists(d + '/train.csv'))
print('MODEL:', MODEL_PATH); print('DATA:', DATA_DIR)

# ---- config: TỐI ƯU VRAM TUYỆT ĐỐI CHỐNG OOM ----
LABELS = ['BENIGN', 'JAILBREAK', 'PROMPT_INJECTION']
L2I = {l: i for i, l in enumerate(LABELS)}; I2L = {i: l for l, i in L2I.items()}

MAXLEN = 256          # 256 bao quát >95% văn bản mà tiết kiệm 40% VRAM so với 320
EPOCHS = 3
LR = 1.5e-5
MICRO_BS = 4          # Batch nhỏ mỗi bước forward để không bao giờ tràn VRAM
ACCUM_STEPS = 4       # Tích lũy 4 bước -> Effective batch size = 16
EVAL_BS = 16          # Batch an toàn khi đánh giá

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Train shape: {train.shape}, Test shape: {test.shape}')
print(train.label.value_counts(normalize=True))

def fmt(df):
    return ('[' + df['language'].astype(str) + '] ' + df['text'].astype(str)).tolist()

# Stratify theo cả label lẫn language để đại diện đồng đều
strat = train.label + '_' + train.language
tr, va = train_test_split(train, test_size=0.15, stratify=strat, random_state=SEED)
tok = AutoTokenizer.from_pretrained(MODEL_PATH)

def encode(texts):
    return tok(texts, truncation=True, max_length=MAXLEN)['input_ids']

X_tr, y_tr = encode(fmt(tr)), tr.label.map(L2I).values
X_va, y_va = encode(fmt(va)), va.label.map(L2I).values
X_te = encode(fmt(test))

cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_tr)
print('Class weights:', cw)

In [ ]:
PAD = tok.pad_token_id

def collate(ids_list):
    m = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), m), PAD, dtype=torch.long)
    att = torch.zeros((len(ids_list), m), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1
    return ids, att

@torch.no_grad()
def predict_logits(m, X, bs=EVAL_BS):
    m.eval()
    order = np.argsort([len(x) for x in X])
    out = np.zeros((len(X), 3), dtype=np.float32)
    for s in range(0, len(X), bs):
        idx = order[s:s+bs]
        ids, att = collate([X[i] for i in idx])
        with torch.autocast('cuda', dtype=torch.float16, enabled=NGPU > 0):
            lg = m(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits
        out[idx] = lg.float().cpu().numpy()
    return out

In [ ]:
# Dọn sạch VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('>>> Loading model weights...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=3, id2label=I2L, label2id=L2I
).to(DEVICE)

# BẬT GRADIENT CHECKPOINTING: Giảm 75% bộ nhớ activation, chống OOM 100%
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Freeze the word-embedding matrix (250k x 1024 = ~256M params, almost half of the model).
# Saves ~3 GB of gradients + AdamW state with negligible effect on accuracy.
model.roberta.embeddings.word_embeddings.weight.requires_grad = False
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_train/1e6:.0f}M | GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

no_decay = ['bias', 'LayerNorm.weight']
params = [
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
opt = torch.optim.AdamW(params, lr=LR)

effective_bs = MICRO_BS * ACCUM_STEPS
steps_per_epoch = (len(X_tr) + effective_bs - 1) // effective_bs
total_steps = steps_per_epoch * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)

scaler = torch.amp.GradScaler('cuda', enabled=NGPU > 0)
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float).to(DEVICE))

best_f1, best_state = -1.0, None
t0 = time.time()

print(f'>>> Bắt đầu huấn luyện (Effective BS = {effective_bs}, Epochs = {EPOCHS})...')
for ep in range(EPOCHS):
    model.train()
    perm = np.random.permutation(len(X_tr))
    run_loss = 0.0
    opt.zero_grad(set_to_none=True)
    
    num_micro_batches = len(perm) // MICRO_BS
    for k in range(num_micro_batches):
        idx = perm[k*MICRO_BS : (k+1)*MICRO_BS]
        ids, att = collate([X_tr[i] for i in idx])
        y = torch.tensor(y_tr[idx]).to(DEVICE)
        
        with torch.autocast('cuda', dtype=torch.float16, enabled=NGPU > 0):
            logits = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits
            loss = loss_fn(logits.float(), y) / ACCUM_STEPS
            
        scaler.scale(loss).backward()
        run_loss += loss.item() * ACCUM_STEPS
        
        if (k + 1) % ACCUM_STEPS == 0 or (k + 1) == num_micro_batches:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
            
            step_num = (k + 1) // ACCUM_STEPS
            if step_num % 50 == 0:
                print(f'Epoch {ep+1}/{EPOCHS} | Step {step_num}/{steps_per_epoch} | Loss: {run_loss / (50 * ACCUM_STEPS):.4f} | [{time.time()-t0:.0f}s]')
                run_loss = 0.0
                if step_num == 50 and ep == 0:
                    print(f'    peak GPU mem so far: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
                
    # Đánh giá sau mỗi epoch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    lg = predict_logits(model, X_va)
    f1 = f1_score(y_va, lg.argmax(1), average='macro')
    per_class = f1_score(y_va, lg.argmax(1), average=None).round(4)
    print(f'=== Hết Epoch {ep+1}: Val Macro-F1 = {f1:.4f} | Per-class: {per_class} ===')
    
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'--> Lưu model tốt nhất với Macro-F1 = {best_f1:.4f}')

print(f'>>> Huấn luyện hoàn tất! Best Val Macro-F1: {best_f1:.4f}')

In [ ]:
# Load trọng số tốt nhất
model.load_state_dict(best_state)
va_logits = predict_logits(model, X_va)
print(classification_report(y_va, va_logits.argmax(1), target_names=LABELS, digits=4))

# Hiển thị F1 theo từng ngôn ngữ
va_pred = va_logits.argmax(1)
print('\nMacro-F1 theo từng ngôn ngữ trên tập Validation:')
for lang in sorted(va.language.unique()):
    m = (va.language == lang).values
    print(f'  {lang} (n={m.sum()}): Macro-F1 = {f1_score(y_va[m], va_pred[m], average="macro"):.4f}')

# Tinh chỉnh nhẹ Logit Bias để tối đa hóa Macro-F1 trên Validation
best_b, best_s = np.zeros(3), f1_score(y_va, va_logits.argmax(1), average='macro')
for b1 in np.arange(-1.0, 1.01, 0.25):
    for b2 in np.arange(-1.0, 1.51, 0.25):
        b = np.array([0.0, b1, b2])
        s = f1_score(y_va, (va_logits + b).argmax(1), average='macro')
        if s > best_s + 1e-4:
            best_s, best_b = s, b
print(f'\nLogit bias tối ưu: {best_b} | Validation Macro-F1 sau tinh chỉnh: {best_s:.4f}')

In [ ]:
print('>>> Đang dự đoán trên tập Test (6,000 dòng)...')
te_logits = predict_logits(model, X_te)
pred = (te_logits + best_b).argmax(1)

sub = pd.DataFrame({'id': test['id'], 'label': [I2L[p] for p in pred]})
sub.to_csv('submission.csv', index=False)

print(f'Shape file nộp: {sub.shape}')
print('Phân bố nhãn dự đoán trên Test:')
print(sub.label.value_counts(normalize=True))

assert len(sub) == 6000, 'Lỗi: Tập test phải có đúng 6000 dòng!'
assert list(sub.columns) == ['id', 'label'], 'Lỗi: Tên cột phải là id,label!'
print('\n✅ File submission.csv đã được tạo thành công! Sẵn sàng để nộp!')
sub.head(10)